custom.geo.json and countries.geojson, to process GeoJSON files and extract centroid coordinates.

# Step 1: Add Imports

In [1]:
import geopandas as gpd
from shapely.geometry import shape, Point
import pandas as pd

# Step 2: Add Helper Functions
Add these functions helpers:

In [ ]:
def load_geojson_from_file(filepath: str) -> Optional[Dict]:
    """Load GeoJSON from a local file."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading GeoJSON from {filepath}: {e}")
        return None

def extract_centroid_from_geometry(geometry: Dict) -> Optional[Dict[str, float]]:
    """
    Extract centroid from a GeoJSON geometry.
    Returns {'latitude': lat, 'longitude': lon} or None.
    """
    try:
        # Create shapely geometry from GeoJSON
        geom = shape(geometry)
        
        # Get centroid
        centroid = geom.centroid
        
        return {
            'latitude': centroid.y,
            'longitude': centroid.x
        }
    except Exception as e:
        print(f"Error extracting centroid: {e}")
        return None

def process_country_geojson(geojson_data: Dict) -> List[Dict[str, Any]]:
    """
    Process country GeoJSON and extract centroids for each country.
    Returns list of country geo point records.
    """
    features = geojson_data.get('features', [])
    country_points = []
    
    for feature in features:
        props = feature.get('properties', {})
        geometry = feature.get('geometry')
        
        if not geometry:
            continue
        
        # Extract ISO codes (try multiple possible field names)
        iso = (
            props.get('ISO3166-1-Alpha-3') or 
            props.get('iso_a3') or 
            props.get('ISO3') or 
            props.get('iso3') or
            props.get('ISO3166-1-Alpha-2') or
            props.get('iso_a2') or
            props.get('ISO2') or
            props.get('iso2')
        )
        
        # Get country name
        country_name = (
            props.get('name') or 
            props.get('NAME') or 
            props.get('NAME_EN') or
            props.get('NAME_LONG') or
            props.get('NAME_ENG') or
            props.get('admin')
        )
        
        # Get continent and subregion if available
        continent = props.get('continent') or props.get('CONTINENT')
        subregion = props.get('subregion') or props.get('SUBREGION')
        
        if not iso or not country_name:
            continue
        
        # Clean ISO
        iso_clean = str(iso).strip().upper()
        
        # Extract centroid
        centroid = extract_centroid_from_geometry(geometry)
        
        if centroid:
            country_points.append({
                'iso': iso_clean,
                'country_name': country_name,
                'continent': continent,
                'subregion': subregion,
                'latitude': centroid['latitude'],
                'longitude': centroid['longitude']
            })
        else:
            print(f"Could not extract centroid for {country_name} ({iso_clean})")
    
    return country_points

def merge_country_geo_points(geo_points_list: List[List[Dict[str, Any]]]) -> List[Dict[str, Any]]:
    """
    Merge multiple geo point sources, preferring first occurrence.
    """
    merged = {}
    
    for geo_points in geo_points_list:
        for point in geo_points:
            iso = point['iso']
            if iso not in merged:
                merged[iso] = point
    
    return list(merged.values())

def normalize_country_geo_points(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Normalize country geo points for database insertion."""
    normalized = []
    for r in rows:
        normalized.append({
            'iso': r['iso'],
            'country_name': r['country_name'],
            'continent': r.get('continent'),
            'subregion': r.get('subregion'),
            'latitude': _to_float(r.get('latitude')),
            'longitude': _to_float(r.get('longitude'))
        })
    return normalized

def get_country_iso_mapping(canonical_rows: List[Dict[str, Any]]) -> Dict[str, str]:
    """Create mapping from ISO to country name from canonical table."""
    return {r['iso']: r['country_name'] for r in canonical_rows}



# Check if geopandas is available
try:
    import geopandas as gpd
    from shapely.geometry import shape, Point
    HAS_GEOPANDAS = True
except ImportError:
    print("Warning: geopandas or shapely not installed. GeoJSON processing will be disabled.")
    print("Install with: pip install geopandas shapely")
    HAS_GEOPANDAS = False
    
def main():
    # Core data downloads (existing code)
    canonical_raw = safe_download(JSON_BUCKET, FILE_MAP["canonical_country_table"], "canonical_country_table")
    # ... rest of your existing downloads ...
    
    if not canonical_raw:
        raise ValueError("canonical_country_table.json is required")
    
    canonical_rows = normalize_canonical_country_table(canonical_raw)
    
    # Process GeoJSON files
    country_geo_points_rows = []
    
    if HAS_GEOPANDAS:
        # Load and process custom.geo.json
        custom_geo_path = "custom.geo.json"  # Adjust path as needed
        if os.path.exists(custom_geo_path):
            custom_geo = load_geojson_from_file(custom_geo_path)
            if custom_geo:
                custom_points = process_country_geojson(custom_geo)
                print(f"Processed {len(custom_points)} countries from custom.geo.json")
                country_geo_points_rows.extend(custom_points)
        
        # Load and process countries.geojson
        countries_geo_path = "countries.geojson"  # Adjust path as needed
        if os.path.exists(countries_geo_path):
            countries_geo = load_geojson_from_file(countries_geo_path)
            if countries_geo:
                countries_points = process_country_geojson(countries_geo)
                print(f"Processed {len(countries_points)} countries from countries.geojson")
                country_geo_points_rows.extend(countries_points)
        
        # Merge points (prioritize first file over second if duplicates)
        if country_geo_points_rows:
            country_geo_points_rows = merge_country_geo_points([country_geo_points_rows])
            print(f"Merged to {len(country_geo_points_rows)} unique country geo points")
            
            # Create ISO to country name mapping from canonical table
            iso_to_name = get_country_iso_mapping(canonical_rows)
            
            # Filter to only ISOs that exist in canonical table
            valid_isos = extract_iso_set(canonical_rows)
            country_geo_points_rows = [
                p for p in country_geo_points_rows 
                if p['iso'] in valid_isos
            ]
            
            # Optionally, fill missing country names from canonical table
            for point in country_geo_points_rows:
                if not point['country_name'] and point['iso'] in iso_to_name:
                    point['country_name'] = iso_to_name[point['iso']]
            
            print(f"Keeping {len(country_geo_points_rows)} geo points with valid ISOs")
            
            # Normalize for database
            country_geo_points_rows = normalize_country_geo_points(country_geo_points_rows)
    
    # Continue with your existing processing...
    country_year_rows = normalize_country_year_features(country_year_raw or [])
    trend_rows = normalize_trend_summary_country(trend_raw or [])
    # ... rest of your processing ...
    
    # Get valid ISOs from canonical table
    valid_isos = extract_iso_set(canonical_rows)
    print(f"canonical_country_table valid ISO count: {len(valid_isos)}")
    
    # Filter all data by valid ISOs (existing code)
    country_year_rows = filter_rows_by_valid_iso(country_year_rows, valid_isos, "country_year_features")
    trend_rows = filter_rows_by_valid_iso(trend_rows, valid_isos, "trend_summary_country")
    forecast_rows = filter_rows_by_valid_iso(forecast_rows, valid_isos, "forecasts_country")
    ranking_rows = filter_rows_by_valid_iso(ranking_rows, valid_isos, "country_rankings")
    top_at_risk_rows = filter_rows_by_valid_iso(top_at_risk_rows, valid_isos, "top_at_risk_countries")
    top_improving_rows = filter_rows_by_valid_iso(top_improving_rows, valid_isos, "top_improving_countries")
    
    # Load in dependency order
    upsert_rows("canonical_country_table", canonical_rows, on_conflict="iso")
    
    # Load country geo points (new table)
    if country_geo_points_rows:
        upsert_rows("country_geo_points", country_geo_points_rows, on_conflict="iso")
    
    # Continue with rest of your tables...
    upsert_rows("country_year_features", country_year_rows, on_conflict="iso,year")
    upsert_rows("trend_summary_country", trend_rows, on_conflict="iso")
    upsert_rows("forecasts_country", forecast_rows, on_conflict="iso")
    upsert_rows("country_rankings", ranking_rows, on_conflict="iso")
    upsert_rows("region_aggregates", region_rows, on_conflict="entity_level,entity_name,year")
    
    # Export/ranking support tables
    upsert_rows("top_at_risk_countries", top_at_risk_rows, on_conflict="iso")
    upsert_rows("top_improving_countries", top_improving_rows, on_conflict="iso")
    upsert_rows("top_worsening_subregions", top_worsening_subregions_rows, on_conflict="subregion")
    upsert_rows("continent_watchlists", continent_watchlists_rows, on_conflict="continent")
    
    print("Done loading Supabase tables.")
    
def ensure_country_geo_points_table():
    """Create the country_geo_points table if it doesn't exist."""
    try:
        supabase.table("country_geo_points").select("iso").limit(1).execute()
        print("country_geo_points table already exists")
    except Exception as e:
        print("Creating country_geo_points table...")
        # Note: You'll need to run this SQL manually in Supabase SQL editor
        # or use supabase's SQL execution if you have that permission
        create_table_sql = """
        create table if not exists public.country_geo_points (
          iso text primary key,
          country_name text,
          continent text,
          subregion text,
          latitude double precision,
          longitude double precision
        );
        """
        # If you have execute_sql capability:
        # supabase.rpc('exec_sql', {'query': create_table_sql}).execute()
        print("Please ensure the table exists in Supabase:")
        print(create_table_sql)
        
def main():
    ensure_country_geo_points_table()
    # ... rest of your code ...